# Fixed-Budget Manual-Review Triage

This notebook documents the uncertainty-aware triage analysis. It compares low-confidence and alternative review-ranking policies at a fixed 20 percent manual-review budget against random selection, using both silver-label and completed manual-audit evidence.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, completed manual-audit records, row-level predictions, bootstrap replicates, or private run artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the fixed-budget triage protocol

Imports analysis and plotting utilities, defines the restricted project-root placeholder, identifies the repeated-seed and manual-audit inputs, fixes the 20 percent review budget, and declares ranking policies, reference targets, and simulation settings.


In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
R1_08_ROOT = ROUND1_DIR / 'artifacts' / 'r1_08_repeated_seed_evaluation_v1'
HUMAN_DIR = ROUND1_DIR / 'reports' / 'human_label_audit' / 'r1_08_repeated_seed'
OUTPUT_DIR = ROUND1_DIR / 'reports' / 'triage_analysis' / 'r1_08_repeated_seed'
FIGURE_DIR = ROUND1_DIR / 'figures' / 'triage_analysis' / 'r1_08_repeated_seed'

SILVER_PRED_PATH = R1_08_ROOT / 'locked_test_predictions_per_model_long_with_silver.csv'
R1_08_MANIFEST_PATH = R1_08_ROOT / 'run_manifest.json'
R1_08_STATE_PATH = R1_08_ROOT / 'run_state.json'
R1_08_CONTRACT_PATH = R1_08_ROOT / 'prediction_contract.json'
HUMAN_PRED_PATH = HUMAN_DIR / 'human_label_evaluation_predictions_per_seed_private.csv'
HUMAN_SUMMARY_PATH = HUMAN_DIR / 'human_label_evaluation_summary.json'

EXPECTED_SEEDS = [42, 52, 62]
LABEL_ORDER = ['negatif', 'netral', 'positif']
PROB_COLUMNS = ['prob_negatif', 'prob_netral', 'prob_positif']
MODEL_ORDER = [
    ('svm', 'SVM'),
    ('indobert', 'IndoBERT'),
    ('wa_svm_bert', 'WA (SVM+BERT)'),
    ('xgboost', 'XGBoost'),
    ('calibrated_xgboost', 'Calibrated XGBoost'),
]
MODEL_NAME = dict(MODEL_ORDER)
MANUAL_REVIEW_BUDGET = 0.20
N_RANDOM_SIMULATIONS = 5000
RANDOM_BASELINE_SEED = 20261031
PROTOCOL_VERSION = 'r1_03_protocol_v2_triage_baselines_human_audit_wa_figures'

TRIAGE_CONFIGS = [
    {'config_key': 'svm_low_confidence', 'model_key': 'svm', 'score_key': 'confidence', 'score_label': 'Lowest maximum probability', 'direction': 'low_first', 'tie_break': 'row_id'},
    {'config_key': 'indobert_low_confidence', 'model_key': 'indobert', 'score_key': 'confidence', 'score_label': 'Lowest maximum probability', 'direction': 'low_first', 'tie_break': 'row_id'},
    {'config_key': 'wa_low_confidence', 'model_key': 'wa_svm_bert', 'score_key': 'confidence', 'score_label': 'Lowest maximum probability', 'direction': 'low_first', 'tie_break': 'row_id'},
    {'config_key': 'xgboost_low_confidence', 'model_key': 'xgboost', 'score_key': 'confidence', 'score_label': 'Lowest maximum probability', 'direction': 'low_first', 'tie_break': 'row_id'},
    {'config_key': 'calibrated_xgboost_low_confidence', 'model_key': 'calibrated_xgboost', 'score_key': 'confidence', 'score_label': 'Lowest maximum probability', 'direction': 'low_first', 'tie_break': 'row_id'},
    {'config_key': 'calibrated_xgboost_high_entropy', 'model_key': 'calibrated_xgboost', 'score_key': 'entropy', 'score_label': 'Highest predictive entropy', 'direction': 'high_first', 'tie_break': 'row_id'},
    {'config_key': 'calibrated_xgboost_low_margin', 'model_key': 'calibrated_xgboost', 'score_key': 'margin', 'score_label': 'Lowest top-two probability margin', 'direction': 'low_first', 'tie_break': 'row_id'},
    {'config_key': 'calibrated_xgboost_base_disagreement', 'model_key': 'calibrated_xgboost', 'score_key': 'base_disagreement', 'score_label': 'SVM-IndoBERT hard-label disagreement', 'direction': 'high_first', 'tie_break': 'lower Calibrated XGBoost confidence, then row_id'},
]

TARGET_SPECS = [
    ('locked_test_silver', 'y_silver_label', 'unweighted', None),
    ('human_audit_manual', 'final_manual_label', 'unweighted', None),
    ('human_audit_manual', 'final_manual_label', 'design_weighted', 'design_weight'),
]

print('Output directory:', OUTPUT_DIR)
print('Seeds:', EXPECTED_SEEDS, '| manual-review budget:', MANUAL_REVIEW_BUDGET, '| random simulations:', N_RANDOM_SIMULATIONS)


### Define ranking, random-baseline, and risk-coverage utilities

Defines checksum and atomic-write helpers, score extraction, deterministic review ordering, random-review simulation, design-weighted calculations, risk-coverage curves, and AURC summaries.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    temporary = path.with_name(f'.{path.name}.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8')
    temporary.replace(path)


def atomic_json(path: Path, payload: dict) -> None:
    temporary = path.with_name(f'.{path.name}.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=True, default=float), encoding='utf-8')
    temporary.replace(path)


def validate_probabilities(values: np.ndarray, name: str) -> None:
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[1] != 3 or not np.isfinite(values).all():
        raise ValueError(f'{name}: invalid probability matrix.')
    if values.min() < -1e-10 or values.max() > 1.0 + 1e-10:
        raise ValueError(f'{name}: probabilities are outside [0, 1].')
    if np.abs(values.sum(axis=1) - 1.0).max() > 1e-5:
        raise ValueError(f'{name}: probabilities do not sum to one.')


def entropy(probabilities: np.ndarray) -> np.ndarray:
    probabilities = np.clip(np.asarray(probabilities, dtype=float), 1e-12, 1.0)
    return -np.sum(probabilities * np.log(probabilities), axis=1)


def top_two_margin(probabilities: np.ndarray) -> np.ndarray:
    ordered = np.sort(np.asarray(probabilities, dtype=float), axis=1)
    return ordered[:, -1] - ordered[:, -2]


def make_seed_wide(long_frame: pd.DataFrame, seed: int) -> pd.DataFrame:
    seed_frame = long_frame.loc[long_frame['seed'].eq(seed)].copy()
    wide = None
    for model_key, _ in MODEL_ORDER:
        group = seed_frame.loc[seed_frame['model_key'].eq(model_key)].sort_values('row_id').reset_index(drop=True)
        if len(group) == 0 or group['row_id'].duplicated().any():
            raise RuntimeError(f'Seed {seed}/{model_key}: invalid row alignment.')
        probabilities = group[PROB_COLUMNS].to_numpy(dtype=float)
        validate_probabilities(probabilities, f'seed {seed}/{model_key}')
        if not np.array_equal(group['prediction'].to_numpy(dtype=int), probabilities.argmax(axis=1)):
            raise RuntimeError(f'Seed {seed}/{model_key}: hard predictions differ from probability argmax.')
        if not np.allclose(group['confidence'].to_numpy(dtype=float), probabilities.max(axis=1), atol=1e-8):
            raise RuntimeError(f'Seed {seed}/{model_key}: confidence differs from maximum probability.')
        if not np.allclose(group['entropy'].to_numpy(dtype=float), entropy(probabilities), atol=1e-8):
            raise RuntimeError(f'Seed {seed}/{model_key}: entropy differs from probability calculation.')
        if wide is None:
            wide = group[['row_id', 'y_silver_label']].copy()
        elif not np.array_equal(wide['row_id'].to_numpy(), group['row_id'].to_numpy()) or not wide['y_silver_label'].eq(group['y_silver_label']).all():
            raise RuntimeError(f'Seed {seed}: models do not share identical row order or silver labels.')
        wide[f'prediction__{model_key}'] = group['prediction'].to_numpy(dtype=int)
        wide[f'prediction_label__{model_key}'] = group['prediction_label'].astype(str).to_numpy()
        wide[f'confidence__{model_key}'] = probabilities.max(axis=1)
        wide[f'entropy__{model_key}'] = entropy(probabilities)
        wide[f'margin__{model_key}'] = top_two_margin(probabilities)
    wide['base_disagreement'] = (wide['prediction__svm'] != wide['prediction__indobert']).astype(int)
    return wide


def review_order(wide: pd.DataFrame, config: dict) -> pd.DataFrame:
    model_key, score_key = config['model_key'], config['score_key']
    work = wide[['row_id']].copy()
    if score_key == 'confidence':
        work['score'] = wide[f'confidence__{model_key}']
        return work.sort_values(['score', 'row_id'], ascending=[True, True], kind='mergesort')
    if score_key == 'entropy':
        work['score'] = wide[f'entropy__{model_key}']
        return work.sort_values(['score', 'row_id'], ascending=[False, True], kind='mergesort')
    if score_key == 'margin':
        work['score'] = wide[f'margin__{model_key}']
        return work.sort_values(['score', 'row_id'], ascending=[True, True], kind='mergesort')
    if score_key == 'base_disagreement':
        work['score'] = wide['base_disagreement']
        work['tie_confidence'] = wide['confidence__calibrated_xgboost']
        return work.sort_values(['score', 'tie_confidence', 'row_id'], ascending=[False, True, True], kind='mergesort')
    raise ValueError(f'Unsupported score: {score_key}')


def evaluate_selection(wide: pd.DataFrame, selected_ids: set[int], config: dict, reference_target: str, target_column: str, weighting: str, weight_column: str | None) -> dict:
    reference = wide.loc[wide[target_column].notna()].copy()
    if reference.empty:
        raise RuntimeError(f'{reference_target}: no reference labels are available.')
    weights = np.ones(len(reference), dtype=float) if weight_column is None else reference[weight_column].to_numpy(dtype=float)
    if not np.isfinite(weights).all() or np.any(weights <= 0):
        raise ValueError(f'{reference_target}: invalid weights.')
    correct = reference[f'prediction_label__{config["model_key"]}'].eq(reference[target_column]).to_numpy()
    manual = reference['row_id'].isin(selected_ids).to_numpy()
    automatic = ~manual
    errors = ~correct
    total_weight = float(weights.sum())
    manual_weight = float(weights[manual].sum())
    automatic_weight = float(weights[automatic].sum())
    total_error_weight = float(weights[errors].sum())
    captured_error_weight = float(weights[errors & manual].sum())
    automatic_correct_weight = float(weights[correct & automatic].sum())
    return {
        'reference_target': reference_target,
        'weighting': weighting,
        'model_key': config['model_key'],
        'model': MODEL_NAME[config['model_key']],
        'config_key': config['config_key'],
        'score_key': config['score_key'],
        'score_label': config['score_label'],
        'selection_direction': config['direction'],
        'tie_break': config['tie_break'],
        'global_locked_test_n': int(len(wide)),
        'global_manual_review_n': int(len(selected_ids)),
        'global_manual_review_budget': float(len(selected_ids) / len(wide)),
        'reference_n': int(len(reference)),
        'reference_weight_total': total_weight,
        'n_manual_review_in_reference': int(manual.sum()),
        'manual_review_rate_in_reference': float(manual.mean()),
        'n_automatic_in_reference': int(automatic.sum()),
        'automatic_coverage': float(automatic.mean()),
        'manual_review_weight': manual_weight,
        'manual_review_weight_rate': float(manual_weight / total_weight),
        'overall_accuracy': float(np.average(correct, weights=weights)),
        'total_errors': int(errors.sum()),
        'total_error_weight': total_error_weight,
        'errors_captured_in_manual_review': int((errors & manual).sum()),
        'captured_error_weight': captured_error_weight,
        'error_capture_rate': float(captured_error_weight / total_error_weight) if total_error_weight else np.nan,
        'manual_review_error_rate': float(captured_error_weight / manual_weight) if manual_weight else np.nan,
        'automatic_accuracy': float(automatic_correct_weight / automatic_weight) if automatic_weight else np.nan,
        'selective_risk': float(1.0 - automatic_correct_weight / automatic_weight) if automatic_weight else np.nan,
    }


def make_random_selection_indices(n_rows: int, n_review: int) -> np.ndarray:
    rng = np.random.default_rng(RANDOM_BASELINE_SEED)
    return np.vstack([rng.choice(n_rows, size=n_review, replace=False) for _ in range(N_RANDOM_SIMULATIONS)]).astype(np.int32, copy=False)


def random_baseline_summary(wide: pd.DataFrame, random_indices: np.ndarray, model_key: str, reference_target: str, target_column: str, weighting: str, weight_column: str | None) -> dict:
    reference_mask = wide[target_column].notna().to_numpy()
    weights = np.zeros(len(wide), dtype=float)
    weights[reference_mask] = 1.0 if weight_column is None else wide.loc[reference_mask, weight_column].to_numpy(dtype=float)
    correct = np.zeros(len(wide), dtype=bool)
    correct[reference_mask] = wide.loc[reference_mask, f'prediction_label__{model_key}'].eq(wide.loc[reference_mask, target_column]).to_numpy()
    correct_weight = weights * correct
    error_weight = weights * (~correct)
    total_weight = weights.sum()
    total_correct_weight = correct_weight.sum()
    total_error_weight = error_weight.sum()
    selected_weight = weights[random_indices].sum(axis=1)
    selected_correct_weight = correct_weight[random_indices].sum(axis=1)
    selected_error_weight = error_weight[random_indices].sum(axis=1)
    selected_n_reference = reference_mask[random_indices].sum(axis=1)
    auto_weight = total_weight - selected_weight
    automatic_accuracy = (total_correct_weight - selected_correct_weight) / auto_weight
    metrics = {
        'manual_review_rate_in_reference': selected_n_reference / reference_mask.sum(),
        'manual_review_weight_rate': selected_weight / total_weight,
        'error_capture_rate': selected_error_weight / total_error_weight if total_error_weight else np.full(N_RANDOM_SIMULATIONS, np.nan),
        'manual_review_error_rate': selected_error_weight / selected_weight,
        'automatic_accuracy': automatic_accuracy,
        'selective_risk': 1.0 - automatic_accuracy,
    }
    row = {
        'reference_target': reference_target, 'weighting': weighting, 'model_key': model_key, 'model': MODEL_NAME[model_key],
        'n_random_simulations': N_RANDOM_SIMULATIONS, 'random_seed': RANDOM_BASELINE_SEED,
        'global_locked_test_n': int(len(wide)), 'global_manual_review_n': int(random_indices.shape[1]),
        'global_manual_review_budget': float(random_indices.shape[1] / len(wide)), 'reference_n': int(reference_mask.sum()),
    }
    for name, values in metrics.items():
        values = np.asarray(values, dtype=float)
        row[f'{name}_mean'] = float(np.nanmean(values))
        row[f'{name}_sample_sd'] = float(np.nanstd(values, ddof=1))
        row[f'{name}_p2_5'] = float(np.nanquantile(values, 0.025))
        row[f'{name}_p97_5'] = float(np.nanquantile(values, 0.975))
    return row


### Load and validate triage inputs

Checks the repeated-seed prediction contract, manual-audit linkage, row and model consistency, label-blind prediction provenance, and the expected three completed seeds before triage is evaluated.


In [ ]:
required_paths = [SILVER_PRED_PATH, R1_08_MANIFEST_PATH, R1_08_STATE_PATH, R1_08_CONTRACT_PATH, HUMAN_PRED_PATH, HUMAN_SUMMARY_PATH]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing triage input(s): ' + '; '.join(missing))

r1_08_manifest = json.loads(R1_08_MANIFEST_PATH.read_text(encoding='utf-8'))
r1_08_state = json.loads(R1_08_STATE_PATH.read_text(encoding='utf-8'))
r1_08_contract = json.loads(R1_08_CONTRACT_PATH.read_text(encoding='utf-8'))
human_summary = json.loads(HUMAN_SUMMARY_PATH.read_text(encoding='utf-8'))
if r1_08_state.get('status') != 'completed' or r1_08_manifest.get('locked_test_metrics_computed') is not True:
    raise RuntimeError('Repeated-seed evaluation must be completed before triage.')
if r1_08_contract.get('seeds') != EXPECTED_SEEDS:
    raise RuntimeError('Repeated-seed seed contract differs from triage input.')
if [(item.get('model_key'), item.get('model')) for item in r1_08_contract.get('model_order', [])] != MODEL_ORDER:
    raise RuntimeError('Repeated-seed model contract differs from triage input.')
if r1_08_manifest.get('output_checksums', {}).get(SILVER_PRED_PATH.name) != sha256_file(SILVER_PRED_PATH):
    raise RuntimeError('The repeated-seed prediction checksum differs from its manifest.')
if human_summary.get('prediction_source_sha256') != sha256_file(SILVER_PRED_PATH):
    raise RuntimeError('The human audit is not linked to the current repeated-seed predictions.')

silver_long = pd.read_csv(SILVER_PRED_PATH)
human_long = pd.read_csv(HUMAN_PRED_PATH)
required_silver = {'seed', 'row_id', 'y_silver_label', 'model_key', 'model', 'prediction', 'prediction_label', 'confidence', 'entropy', *PROB_COLUMNS}
required_human = {'seed', 'row_id', 'final_manual_label', 'silver_label_lstm', 'design_weight', 'model_key', 'prediction_label'}
if missing_columns := sorted(required_silver.difference(silver_long.columns)):
    raise ValueError(f'Repeated-seed prediction columns missing: {missing_columns}')
if missing_columns := sorted(required_human.difference(human_long.columns)):
    raise ValueError(f'Human-label prediction columns missing: {missing_columns}')
if sorted(silver_long['seed'].unique().tolist()) != EXPECTED_SEEDS or set(silver_long['model_key']) != set(MODEL_NAME):
    raise RuntimeError('Repeated-seed predictions have unexpected seeds or models.')
if len(silver_long) != 1172 * len(EXPECTED_SEEDS) * len(MODEL_ORDER) or silver_long.duplicated(['seed', 'row_id', 'model_key']).any():
    raise RuntimeError('Repeated-seed long prediction keys or row count are invalid.')

audit_reference = human_long[['row_id', 'final_manual_label', 'silver_label_lstm', 'design_weight']].drop_duplicates('row_id').copy()
if len(audit_reference) != 300 or audit_reference['row_id'].duplicated().any() or audit_reference.isna().any().any():
    raise RuntimeError('The completed locked-test manual audit must contain exactly 300 valid rows.')
if (audit_reference['design_weight'] <= 0).any():
    raise RuntimeError('Audit design weights must be positive.')

SEED_WIDE = {}
reference_row_ids = None
reference_silver_labels = None
for seed in EXPECTED_SEEDS:
    wide = make_seed_wide(silver_long, seed).sort_values('row_id').reset_index(drop=True)
    wide = wide.merge(audit_reference, on='row_id', how='left', validate='one_to_one')
    audit_mask = wide['final_manual_label'].notna()
    if audit_mask.sum() != 300 or not wide.loc[audit_mask, 'y_silver_label'].eq(wide.loc[audit_mask, 'silver_label_lstm']).all():
        raise RuntimeError(f'Seed {seed}: audit labels do not align with repeated-seed silver labels.')
    if reference_row_ids is None:
        reference_row_ids = wide['row_id'].to_numpy(dtype=int)
        reference_silver_labels = wide['y_silver_label'].to_numpy(dtype=str)
    elif not np.array_equal(reference_row_ids, wide['row_id'].to_numpy(dtype=int)) or not np.array_equal(reference_silver_labels, wide['y_silver_label'].to_numpy(dtype=str)):
        raise RuntimeError('Locked-test rows or silver labels differ between seeds.')
    SEED_WIDE[seed] = wide

GLOBAL_ROW_IDS = reference_row_ids
GLOBAL_REVIEW_N = int(np.ceil(MANUAL_REVIEW_BUDGET * len(GLOBAL_ROW_IDS)))
RANDOM_SELECTION_INDICES = make_random_selection_indices(len(GLOBAL_ROW_IDS), GLOBAL_REVIEW_N)
INPUT_HASHES = {str(path.relative_to(PROJECT_ROOT)): sha256_file(path) for path in required_paths}

print('Validated locked test rows:', len(GLOBAL_ROW_IDS), '| global manual-review rows:', GLOBAL_REVIEW_N)
print('Validated manual audit rows:', len(audit_reference), '| seeds:', EXPECTED_SEEDS)


### Evaluate fixed-budget review-ranking policies

For every seed and policy, selects the 20 percent highest-priority records, measures selected-set error concentration and manual-audit outcomes, and compares each policy with a random-review baseline.


In [ ]:
fixed_rows = []
audit_selection_rows = []
for seed, wide in SEED_WIDE.items():
    for config in TRIAGE_CONFIGS:
        ordered = review_order(wide, config)
        selected_ids = set(ordered.head(GLOBAL_REVIEW_N)['row_id'].astype(int).tolist())
        if len(selected_ids) != GLOBAL_REVIEW_N:
            raise RuntimeError(f'Seed {seed}/{config["config_key"]}: selected review count is invalid.')
        for reference_target, target_column, weighting, weight_column in TARGET_SPECS:
            row = evaluate_selection(wide, selected_ids, config, reference_target, target_column, weighting, weight_column)
            row['seed'] = seed
            fixed_rows.append(row)
        audit = wide.loc[wide['final_manual_label'].notna()].copy()
        audit['seed'] = seed
        audit['config_key'] = config['config_key']
        audit['model_key'] = config['model_key']
        audit['model'] = MODEL_NAME[config['model_key']]
        audit['score_key'] = config['score_key']
        audit['selected_for_manual_review'] = audit['row_id'].isin(selected_ids)
        audit['model_prediction'] = audit[f'prediction_label__{config["model_key"]}']
        audit['model_correct_against_manual'] = audit['model_prediction'].eq(audit['final_manual_label'])
        audit['model_correct_against_silver'] = audit['model_prediction'].eq(audit['y_silver_label'])
        audit['score_value'] = (
            audit[f'confidence__{config["model_key"]}'] if config['score_key'] == 'confidence' else
            audit[f'entropy__{config["model_key"]}'] if config['score_key'] == 'entropy' else
            audit[f'margin__{config["model_key"]}'] if config['score_key'] == 'margin' else
            audit['base_disagreement']
        )
        audit_selection_rows.append(audit[[
            'seed', 'config_key', 'model_key', 'model', 'score_key', 'row_id', 'y_silver_label',
            'final_manual_label', 'design_weight', 'model_prediction', 'model_correct_against_manual',
            'model_correct_against_silver', 'score_value', 'base_disagreement', 'selected_for_manual_review',
        ]])

fixed_budget = pd.DataFrame(fixed_rows).sort_values(['reference_target', 'weighting', 'config_key', 'seed']).reset_index(drop=True)
audit_selection_private = pd.concat(audit_selection_rows, ignore_index=True).sort_values(['seed', 'config_key', 'row_id']).reset_index(drop=True)

alignment_rows = []
low_confidence_audit = audit_selection_private.loc[
    audit_selection_private['config_key'].str.endswith('_low_confidence') &
    audit_selection_private['selected_for_manual_review']
].copy()
for (seed, config_key, model_key, model), group in low_confidence_audit.groupby(['seed', 'config_key', 'model_key', 'model'], sort=True):
    manual_correct = group['model_correct_against_manual'].to_numpy(dtype=bool)
    silver_correct = group['model_correct_against_silver'].to_numpy(dtype=bool)
    n_selected = len(group)
    conditions = {
        'both_correct': manual_correct & silver_correct,
        'manual_only_correct': manual_correct & ~silver_correct,
        'silver_only_correct': ~manual_correct & silver_correct,
        'both_wrong': ~manual_correct & ~silver_correct,
    }
    row = {'seed': seed, 'config_key': config_key, 'model_key': model_key, 'model': model, 'selected_audit_n': n_selected}
    for name, mask in conditions.items():
        row[f'{name}_n'] = int(mask.sum())
        row[f'{name}_rate'] = float(mask.mean())
    if sum(row[f'{name}_n'] for name in conditions) != n_selected:
        raise RuntimeError(f'Human-silver alignment conditions do not partition {seed}/{config_key}.')
    alignment_rows.append(row)
human_silver_alignment = pd.DataFrame(alignment_rows).sort_values(['config_key', 'seed']).reset_index(drop=True)
alignment_summary_rows = []
alignment_rate_columns = ['both_correct_rate', 'manual_only_correct_rate', 'silver_only_correct_rate', 'both_wrong_rate']
for (config_key, model_key, model), group in human_silver_alignment.groupby(['config_key', 'model_key', 'model'], sort=True):
    row = {'config_key': config_key, 'model_key': model_key, 'model': model, 'seeds': ','.join(str(seed) for seed in sorted(group['seed'].tolist()))}
    row['selected_audit_n_mean'] = float(group['selected_audit_n'].mean())
    row['selected_audit_n_sample_sd'] = float(group['selected_audit_n'].std(ddof=1))
    for column in alignment_rate_columns:
        row[f'{column}_mean'] = float(group[column].mean())
        row[f'{column}_sample_sd'] = float(group[column].std(ddof=1))
    alignment_summary_rows.append(row)
human_silver_alignment_summary = pd.DataFrame(alignment_summary_rows).sort_values('config_key').reset_index(drop=True)

random_rows = []
for seed, wide in SEED_WIDE.items():
    for model_key, _ in MODEL_ORDER:
        for reference_target, target_column, weighting, weight_column in TARGET_SPECS:
            row = random_baseline_summary(wide, RANDOM_SELECTION_INDICES, model_key, reference_target, target_column, weighting, weight_column)
            row['seed'] = seed
            random_rows.append(row)
random_baseline = pd.DataFrame(random_rows).sort_values(['reference_target', 'weighting', 'model_key', 'seed']).reset_index(drop=True)

comparison_columns = [
    'reference_target', 'weighting', 'seed', 'model_key',
    'error_capture_rate_mean', 'automatic_accuracy_mean', 'selective_risk_mean',
]
triage_vs_random = fixed_budget.merge(random_baseline[comparison_columns], on=['reference_target', 'weighting', 'seed', 'model_key'], how='left', validate='many_to_one')
triage_vs_random['error_capture_rate_gain_vs_random'] = triage_vs_random['error_capture_rate'] - triage_vs_random['error_capture_rate_mean']
triage_vs_random['automatic_accuracy_gain_vs_random'] = triage_vs_random['automatic_accuracy'] - triage_vs_random['automatic_accuracy_mean']
triage_vs_random['selective_risk_reduction_vs_random'] = triage_vs_random['selective_risk_mean'] - triage_vs_random['selective_risk']

SUMMARY_METRICS = ['manual_review_rate_in_reference', 'error_capture_rate', 'manual_review_error_rate', 'automatic_accuracy', 'selective_risk']
fixed_budget_summary = fixed_budget.groupby([
    'reference_target', 'weighting', 'model_key', 'model', 'config_key', 'score_key', 'score_label', 'selection_direction', 'tie_break'
], as_index=False)[SUMMARY_METRICS].agg(['mean', 'std']).reset_index()
fixed_budget_summary.columns = [
    '_'.join([str(part) for part in column if part]).rstrip('_') if isinstance(column, tuple) else column
    for column in fixed_budget_summary.columns
]

print('Fixed-budget rows:', len(fixed_budget), '| expected:', len(EXPECTED_SEEDS) * len(TRIAGE_CONFIGS) * len(TARGET_SPECS))
print('Random-baseline rows:', len(random_baseline), '| audit selection rows:', len(audit_selection_private), '| human-silver low-confidence summaries:', len(human_silver_alignment))
display(triage_vs_random.loc[
    (triage_vs_random['reference_target'].eq('locked_test_silver')) &
    (triage_vs_random['config_key'].isin(['xgboost_low_confidence', 'calibrated_xgboost_low_confidence']))
][['seed', 'config_key', 'error_capture_rate', 'error_capture_rate_mean', 'error_capture_rate_gain_vs_random', 'automatic_accuracy']])


### Calculate risk-coverage curves and triage figures

Builds risk-coverage and error-capture summaries across manual-review budgets, derives AURC values, and produces aggregate figures for the manuscript and supplementary material.


In [ ]:
def risk_coverage_curve(wide: pd.DataFrame, seed: int, config: dict) -> pd.DataFrame:
    review_first = review_order(wide, config)
    automatic_first = review_first.iloc[::-1].reset_index(drop=True)
    lookup = wide.set_index('row_id')
    automatic = lookup.loc[automatic_first['row_id']]
    errors = ~automatic[f'prediction_label__{config["model_key"]}'].eq(automatic['y_silver_label']).to_numpy()
    accepted_n = np.arange(1, len(automatic) + 1)
    selective_risk = np.cumsum(errors) / accepted_n
    return pd.DataFrame({
        'seed': seed, 'config_key': config['config_key'], 'model_key': config['model_key'],
        'model': MODEL_NAME[config['model_key']], 'score_key': config['score_key'],
        'accepted_automatic_n': accepted_n, 'automatic_coverage': accepted_n / len(automatic),
        'manual_review_budget': 1.0 - accepted_n / len(automatic), 'selective_risk': selective_risk,
    })


risk_rows = []
aurc_rows = []
for seed, wide in SEED_WIDE.items():
    for config in TRIAGE_CONFIGS:
        curve = risk_coverage_curve(wide, seed, config)
        risk_rows.append(curve)
        aurc_rows.append({
            'seed': seed, 'config_key': config['config_key'], 'model_key': config['model_key'],
            'model': MODEL_NAME[config['model_key']], 'score_key': config['score_key'],
            'aurc_discrete_mean_selective_risk': float(curve['selective_risk'].mean()),
            'overall_silver_error_rate': float(curve['selective_risk'].iloc[-1]),
        })
risk_coverage = pd.concat(risk_rows, ignore_index=True)
aurc_summary = pd.DataFrame(aurc_rows).sort_values(['config_key', 'seed']).reset_index(drop=True)

plot_configs = [
    'xgboost_low_confidence', 'calibrated_xgboost_low_confidence', 'wa_low_confidence',
    'calibrated_xgboost_high_entropy', 'calibrated_xgboost_low_margin',
    'calibrated_xgboost_base_disagreement',
]
plot_labels = {
    'xgboost_low_confidence': 'XGBoost confidence',
    'calibrated_xgboost_low_confidence': 'Calibrated XGBoost confidence',
    'wa_low_confidence': 'WA confidence',
    'calibrated_xgboost_high_entropy': 'Calibrated XGBoost entropy',
    'calibrated_xgboost_low_margin': 'Calibrated XGBoost margin',
    'calibrated_xgboost_base_disagreement': 'SVM-IndoBERT disagreement',
}
fig, axes = plt.subplots(1, len(EXPECTED_SEEDS), figsize=(15, 4.6), sharex=True, sharey=True)
for axis, seed in zip(axes, EXPECTED_SEEDS):
    for config_key in plot_configs:
        curve = risk_coverage.loc[(risk_coverage['seed'].eq(seed)) & (risk_coverage['config_key'].eq(config_key))]
        axis.plot(curve['automatic_coverage'], curve['selective_risk'], linewidth=1.6, label=plot_labels[config_key])
    xgb_error = aurc_summary.loc[(aurc_summary['seed'].eq(seed)) & (aurc_summary['config_key'].eq('xgboost_low_confidence')), 'overall_silver_error_rate'].iloc[0]
    cal_error = aurc_summary.loc[(aurc_summary['seed'].eq(seed)) & (aurc_summary['config_key'].eq('calibrated_xgboost_low_confidence')), 'overall_silver_error_rate'].iloc[0]
    axis.axhline(xgb_error, color='tab:blue', linestyle=':', linewidth=1.1, alpha=0.9, label='Random expectation: XGBoost')
    axis.axhline(cal_error, color='black', linestyle=':', linewidth=1.1, alpha=0.9, label='Random expectation: Calibrated XGBoost')
    axis.set_title(f'Seed {seed}')
    axis.set_xlabel('Automatic coverage')
    axis.grid(alpha=0.25)
axes[0].set_ylabel('Selective risk against silver labels')
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.12))
fig.tight_layout(rect=(0, 0.12, 1, 1))
risk_figure_path = FIGURE_DIR / 'triage_risk_coverage_silver_per_seed.png'
fig.savefig(risk_figure_path, dpi=180, bbox_inches='tight')
plt.close(fig)

bar_configs = ['xgboost_low_confidence', 'calibrated_xgboost_low_confidence', 'wa_low_confidence', 'calibrated_xgboost_high_entropy', 'calibrated_xgboost_low_margin', 'calibrated_xgboost_base_disagreement']
bar_silver = fixed_budget_summary.loc[(fixed_budget_summary['reference_target'].eq('locked_test_silver')) & (fixed_budget_summary['weighting'].eq('unweighted')) & (fixed_budget_summary['config_key'].isin(bar_configs))].set_index('config_key').reindex(bar_configs)
bar_human = fixed_budget_summary.loc[(fixed_budget_summary['reference_target'].eq('human_audit_manual')) & (fixed_budget_summary['weighting'].eq('unweighted')) & (fixed_budget_summary['config_key'].isin(bar_configs))].set_index('config_key').reindex(bar_configs)
x = np.arange(len(bar_configs))
fig, axis = plt.subplots(figsize=(12.0, 5.0))
error_style = {'ecolor': '#333333', 'elinewidth': 1.0, 'capsize': 3, 'capthick': 1.0}
axis.bar(x - 0.18, bar_silver['error_capture_rate_mean'], yerr=bar_silver['error_capture_rate_std'], width=0.36, label='Silver-label reference', error_kw=error_style)
axis.bar(x + 0.18, bar_human['error_capture_rate_mean'], yerr=bar_human['error_capture_rate_std'], width=0.36, label='Manual-audit reference', error_kw=error_style)
axis.set_xticks(x, [plot_labels[key] for key in bar_configs], rotation=20, ha='right')
axis.set_ylim(0, 1)
axis.set_ylabel('Error capture rate')
axis.set_title('Error capture under the fixed global manual-review budget (mean +/- sample SD)')
axis.grid(axis='y', alpha=0.25)
axis.legend()
fig.tight_layout()
capture_figure_path = FIGURE_DIR / 'triage_error_capture_by_reference.png'
fig.savefig(capture_figure_path, dpi=180, bbox_inches='tight')
plt.close(fig)

print('Risk-coverage rows:', len(risk_coverage), '| figures:', risk_figure_path.name, ',', capture_figure_path.name)


### Write triage summaries and completion manifest

Writes aggregate policy, random-baseline, error-capture, risk-coverage, and AURC tables plus figures and a semantic completion summary. Private selection lists and row-level audit artifacts remain excluded from .


In [ ]:
output_paths = {
    'triage_fixed_budget_per_seed.csv': fixed_budget,
    'triage_fixed_budget_repeated_seed_summary.csv': fixed_budget_summary,
    'triage_random_baseline_per_seed.csv': random_baseline,
    'triage_vs_random_per_seed.csv': triage_vs_random,
    'triage_risk_coverage_silver_per_seed.csv': risk_coverage,
    'triage_aurc_silver_per_seed.csv': aurc_summary,
    'triage_human_audit_selection_per_seed_private.csv': audit_selection_private,
    'triage_low_confidence_human_silver_alignment_per_seed.csv': human_silver_alignment,
    'triage_low_confidence_human_silver_alignment_repeated_seed_summary.csv': human_silver_alignment_summary,
}
for filename, frame in output_paths.items():
    atomic_csv(OUTPUT_DIR / filename, frame)

expected_fixed = len(EXPECTED_SEEDS) * len(TRIAGE_CONFIGS) * len(TARGET_SPECS)
expected_random = len(EXPECTED_SEEDS) * len(MODEL_ORDER) * len(TARGET_SPECS)
expected_risk = len(EXPECTED_SEEDS) * len(TRIAGE_CONFIGS) * len(GLOBAL_ROW_IDS)
if len(fixed_budget) != expected_fixed or len(random_baseline) != expected_random or len(risk_coverage) != expected_risk:
    raise RuntimeError('Unexpected triage output row count.')
if fixed_budget['global_manual_review_n'].ne(GLOBAL_REVIEW_N).any() or not np.isclose(fixed_budget['global_manual_review_budget'], GLOBAL_REVIEW_N / len(GLOBAL_ROW_IDS)).all():
    raise RuntimeError('The global fixed manual-review budget was not applied consistently.')
if audit_selection_private.groupby(['seed', 'config_key'])['row_id'].nunique().ne(300).any():
    raise RuntimeError('Human-audit selection export is incomplete.')
if len(human_silver_alignment) != len(EXPECTED_SEEDS) * len(MODEL_ORDER):
    raise RuntimeError('Low-confidence human-silver alignment summary is incomplete.')

summary = {
    'status': 'completed',
    'completed_at_utc': utc_now(),
    'seeds': EXPECTED_SEEDS,
    'locked_test_n': int(len(GLOBAL_ROW_IDS)),
    'human_audit_n': int(len(audit_reference)),
    'global_manual_review_n': GLOBAL_REVIEW_N,
    'global_manual_review_budget': float(GLOBAL_REVIEW_N / len(GLOBAL_ROW_IDS)),
    'triage_configs': TRIAGE_CONFIGS,
    'random_baseline': {'n_simulations': N_RANDOM_SIMULATIONS, 'random_seed': RANDOM_BASELINE_SEED, 'selection': 'uniform random global locked-test subset without replacement'},
    'human_audit_interpretation': 'The audit is stratified by LSTM silver label. Unweighted and design-weighted human results are secondary, and neither converts the full locked test to human ground truth.',
    'human_silver_alignment_summary': 'For each low-confidence strategy, selected audit cases are partitioned into both correct, manual-only correct, silver-only correct, and both wrong. These counts are descriptive per seed and are not pooled as independent observations.',
    'guardrails': [
        'No model training, calibration fitting, threshold tuning, seed selection, or probability averaging occurred in this triage analysis.',
        'The 20 percent review policy is a global locked-test ranking. Human-audit coverage can differ from 20 percent because the audit is a 300-row stratified subset.',
        'Silver-label and manual-label triage metrics must be reported separately.',
    ],
    'input_hashes': INPUT_HASHES,
    'output_checksums': {filename: sha256_file(OUTPUT_DIR / filename) for filename in output_paths},
    'figure_paths': [str(risk_figure_path.relative_to(PROJECT_ROOT)), str(capture_figure_path.relative_to(PROJECT_ROOT))],
}
atomic_json(OUTPUT_DIR / 'triage_analysis_plan_and_manifest.json', summary)
atomic_json(OUTPUT_DIR / 'input_integrity_summary.json', {
    'r1_08_prediction_rows': int(len(silver_long)),
    'human_prediction_rows': int(len(human_long)),
    'manual_audit_rows': int(len(audit_reference)),
    'global_locked_test_rows': int(len(GLOBAL_ROW_IDS)),
    'input_hashes': INPUT_HASHES,
})
atomic_json(OUTPUT_DIR / 'r1_03_summary.json', {
    'status': 'completed', 'completed_at_utc': utc_now(), 'fixed_budget_rows': int(len(fixed_budget)),
    'random_baseline_rows': int(len(random_baseline)), 'risk_coverage_rows': int(len(risk_coverage)),
    'low_confidence_human_silver_alignment_rows': int(len(human_silver_alignment)),
    'note': 'Report per-seed values and mean plus sample SD. Do not choose a best seed or pool locked-test rows across seeds.',
})
atomic_json(OUTPUT_DIR / 'run_state.json', {
    'protocol_version': PROTOCOL_VERSION, 'status': 'completed', 'completed_at_utc': utc_now(),
    'completed_stages': ['input_validation', 'fixed_budget_triage', 'random_baseline', 'human_audit_triage', 'human_silver_alignment_summary', 'risk_coverage', 'figures', 'finalization'],
})

print('completed:', OUTPUT_DIR)
print('Interpret silver and manual references separately. Report each seed and mean +/- sample SD without selecting a winner.')
